In [1]:
import torch
import numpy as np
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Module
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam

In [2]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [3]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size) -> None:
        super().__init__()
        self.net = Sequential(
            Linear(input_size, 32),
            ReLU(),
            Linear(32,16),
            ReLU(),
            Linear(16, output_size)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [4]:
def train_model_reg(features, targets, kf, batch, epochs, criterion):
    acc_scores = []
    f1_scores = []
    for i, (train_index, test_index) in enumerate(kf.split(features, targets)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        model = CustomModel(13, 3)
        optmizer = Adam(model.parameters(), lr=0.001)
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(features[test_index]).float()
            y_true = targets[test_index]
            output = model(x_test_tensor)
            result = torch.argmax(output, dim=1).cpu().numpy()
            acc_s = accuracy_score(y_true, result)
            f1_s = f1_score(y_true, result, average="macro")
            cm = confusion_matrix(y_true, result)
            acc_scores.append(acc_s)
            f1_scores.append(f1_s)
            print(f"Acc: {acc_s}")
            print(f"F1-Score: {f1_s}")
            print(f"CM: {cm}")

    print(f"\nResultado modelo ----- Acc: {np.mean(acc_scores)}, F1-Score: {np.mean(f1_scores)}")

In [5]:
dataset = load_wine()
features = dataset.data
targets = dataset.target

In [6]:
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 8
kf = StratifiedKFold(n_splits=5, shuffle=True)

In [7]:
train_model_reg(features, targets, kf, batch, max_epochs, criterion)

Treinando modelo ----- Fold 1
Acc: 0.9166666666666666
F1-Score: 0.9193411952032641
CM: [[10  2  0]
 [ 0 13  1]
 [ 0  0 10]]
Treinando modelo ----- Fold 2
Acc: 0.9444444444444444
F1-Score: 0.945824706694272
CM: [[11  1  0]
 [ 0 13  1]
 [ 0  0 10]]
Treinando modelo ----- Fold 3
Acc: 0.8888888888888888
F1-Score: 0.8850170502983802
CM: [[11  1  0]
 [ 0 14  0]
 [ 0  3  7]]
Treinando modelo ----- Fold 4
Acc: 0.9714285714285714
F1-Score: 0.9740129935032483
CM: [[11  1  0]
 [ 0 14  0]
 [ 0  0  9]]
Treinando modelo ----- Fold 5
Acc: 0.9142857142857143
F1-Score: 0.9168647425014148
CM: [[ 9  2  0]
 [ 0 14  1]
 [ 0  0  9]]

Resultado modelo ----- Acc: 0.9271428571428573, F1-Score: 0.9282121376401158
